### **VideoCLIP con Docker, comparación OpenClip**

Este cuaderno implementa la línea base OpenCLIP usando frames representativos (Fase 1). Luego se extiende a una representación temporal basada en video usando un encoder preentrenado en Colab (Fase 2).

Se ha extraido un suboconjunto de videos del dataset ActivityNet, con el cual se revisa recuperación y negativos difíciles y deja el pipeline automatizado en `scripts/` y `src/`.

#### **Idea principal**
El flujo realista de trabajo es:

1. trabajar primero con un checkpoint preentrenado ==????
2. extraer embeddings de un subconjunto real,
3. evaluar recuperación cruzada,
4. inspeccionar errores???

#### **Fase 1: OpenCLIP (CPU)**

```text
ActivityNet Segment
(start, end, caption)
        │
        ▼
Extraer 5 frames
        │
        ▼
OpenCLIP Image Encoder
        │
        ▼
Frame Embeddings
        │
        ▼
Mean Pooling
        │
        ▼
Clip Embedding
```

```text
Caption
        │
        ▼
OpenCLIP Text Encoder
        │
        ▼
Text Embedding
```

```text
Clip Embedding
        │
        ├───────────────┐
        ▼               ▼
Cosine Similarity  Text Embedding
        │
        ▼
Ranking de segmentos
        │
        ▼
Recall@1, Recall@5, MRR
```

#### **Fase 2: VideoMAE (GPU Colab)

```text
ActivityNet Segment
(start, end)
        │
        ▼
Clip de Video
        │
        ▼
VideoMAE Encoder
        │
        ▼
Video Embedding
```

```text
Caption
        │
        ▼
Text Encoder
        │
        ▼
Text Embedding
```

```text
Video Embedding
        │
        ├───────────────┐
        ▼               ▼
Cosine Similarity  Text Embedding
        │
        ▼
Ranking de segmentos
        │
        ▼
Recall@1, Recall@5, MRR
```

In [5]:
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
import pandas as pd
from tqdm import tqdm

#Extraer 5 frames por segmento
#data/activitynet/frames/

#28GYivx4lyk_seg0_f0.jpg
#28GYivx4lyk_seg0_f1.jpg
def extract_frame_at_time(cap: cv2.VideoCapture, time_sec: float):
    cap.set(cv2.CAP_PROP_POS_MSEC, time_sec * 1000)
    success, frame = cap.read()

    if not success:
        return None

    return frame


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--metadata-csv",
        default="data/activitynet/activitynet_subset.csv",
    )

    parser.add_argument(
        "--videos-dir",
        default="data/activitynet/videos",
    )

    parser.add_argument(
        "--frames-dir",
        default="data/activitynet/frames",
    )

    parser.add_argument(
        "--frames-per-segment",
        type=int,
        default=5,
    )

    parser.add_argument(
        "--output-csv",
        default="data/activitynet/activitynet_frames.csv",
    )

    args = parser.parse_args()

    metadata_csv = Path(args.metadata_csv)
    videos_dir = Path(args.videos_dir)
    frames_dir = Path(args.frames_dir)

    frames_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(metadata_csv)

    records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        video_file = row["video_file"]
        video_path = videos_dir / video_file

        if not video_path.exists():
            print(f"[WARN] No existe: {video_path}")
            continue

        start_time = float(row["start"])
        end_time = float(row["end"])

        duration = max(end_time - start_time, 0.01)

        cap = cv2.VideoCapture(str(video_path))

        if not cap.isOpened():
            print(f"[WARN] No se pudo abrir: {video_path}")
            continue

        for frame_idx in range(args.frames_per_segment):

            if args.frames_per_segment == 1:
                t = (start_time + end_time) / 2
            else:
                alpha = frame_idx / (args.frames_per_segment - 1)
                t = start_time + alpha * duration

            frame = extract_frame_at_time(cap, t)

            if frame is None:
                continue

            image_name = (
                f"{row['youtube_id']}"
                f"_seg{row['segment_id']}"
                f"_f{frame_idx}.jpg"
            )

            image_path = frames_dir / image_name

            cv2.imwrite(str(image_path), frame)

            records.append(
                {
                    "video_id": row["video_id"],
                    "youtube_id": row["youtube_id"],
                    "segment_id": row["segment_id"],
                    "start": start_time,
                    "end": end_time,
                    "caption": row["caption"],
                    "frame_idx": frame_idx,
                    "frame_path": str(image_path.resolve()),
                }
            )

        cap.release()

    out_df = pd.DataFrame(records)
    out_df.to_csv(args.output_csv, index=False)

    print()
    print("Frames extraídos:", len(out_df))
    print("CSV guardado en:", args.output_csv)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--metadata-csv METADATA_CSV]
                             [--videos-dir VIDEOS_DIR]
                             [--frames-dir FRAMES_DIR]
                             [--frames-per-segment FRAMES_PER_SEGMENT]
                             [--output-csv OUTPUT_CSV]
ipykernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-88a6885d-4ac5-4a11-b4b1-0abffef2a408.json


SystemExit: 2

### **python scripts/04_extract_segment_frames.py \
  --metadata-csv data/activitynet/activitynet_subset.csv \
  --videos-dir data/activitynet/videos \
  --frames-dir data/activitynet/frames \
  --frames-per-segment 5 \
  --output-csv data/activitynet/activitynet_frames.csv

In [3]:
#05_build_clip_embeddings
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path(__file__).resolve().parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import argparse
import numpy as np
import pandas as pd

from src.openclip_utils import (
    create_model,
    encode_image_paths,
    encode_texts,
)


def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--frames-csv",
        default="data/activitynet/activitynet_frames.csv",
    )

    parser.add_argument(
        "--model-name",
        default="ViT-B-32",
    )

    parser.add_argument(
        "--pretrained",
        default="laion2b_s34b_b79k",
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--output",
        default="outputs/embeddings/activitynet_clip_embeddings.npz",
    )

    args = parser.parse_args()

    df = pd.read_csv(args.frames_csv)

    model, preprocess, tokenizer, device = create_model(
        args.model_name,
        args.pretrained,
    )

    segment_features = []
    segment_ids = []
    captions = []

    grouped = df.groupby(
        [
            "video_id",
            "segment_id",
        ]
    )

    for (video_id, segment_id), group in grouped:

        frame_paths = group["frame_path"].tolist()

        frame_embeddings = encode_image_paths(
            model,
            preprocess,
            frame_paths,
            device,
            batch_size=args.batch_size,
        )

        segment_embedding = frame_embeddings.mean(axis=0)

        segment_features.append(segment_embedding)

        segment_ids.append(
            f"{video_id}_seg{segment_id}"
        )

        captions.append(
            group.iloc[0]["caption"]
        )

    segment_features = np.vstack(segment_features)

    text_features = encode_texts(
        model,
        tokenizer,
        captions,
        device,
        batch_size=max(args.batch_size, 32),
    )

    output_path = Path(args.output)
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        output_path,
        segment_features=segment_features,
        text_features=text_features,
        segment_ids=np.array(segment_ids),
        captions=np.array(captions),
        model_name=args.model_name,
        pretrained=args.pretrained,
    )

    print()
    print("Embeddings guardados en:")
    print(output_path)

    print()
    print("Segmentos:", len(segment_ids))
    print("Textos:", len(captions))
    print("Dimensión:", segment_features.shape[1])


if __name__ == "__main__":
    main()


root@08ad89565310:/workspace/Semana4/ProyectoFinal# python scripts/05_build_clip_embeddings.py
open_clip_pytorch_model.bin: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 605M/605M [01:04<00:00, 9.40MB/s]

Embeddings guardados en:
outputs/embeddings/activitynet_clip_embeddings.npz

Segmentos: 109
Textos: 109
Dimensión: 512

#### **1. Cargar configuración local**


In [ ]:
metadata = load_metadata(PROJECT_ROOT / "data/bootstrap_flickr30k/metadata.csv", root=PROJECT_ROOT)
metadata[["image_id", "label", "caption"]]


#### **4. Cargar OpenCLIP**

En esta etapa cargamos **OpenCLIP**, una implementación abierta de modelos tipo CLIP entrenados para alinear imágenes y texto en un mismo espacio de representación. La idea central es aprender dos codificadores: uno visual y uno textual. El codificador visual transforma una imagen en un vector y el codificador textual transforma una **caption** o **prompt** en otro vector. Si la imagen y el texto describen el mismo contenido, ambos vectores deberían quedar cercanos según una medida como similitud coseno.

Para este laboratorio usamos un baseline sólido y razonable para una sola GPU:

- `ViT-B-32`
- `laion2b_s34b_b79k`

`ViT-B-32` indica que el encoder visual está basado en un **Vision Transformer** base con parches de tamaño 32. Esto ofrece un equilibrio adecuado entre costo computacional y calidad de representación. No es el modelo más grande, pero sí es suficientemente expresivo para estudiar recuperación imagen-texto en una sesión de laboratorio.

`laion2b_s34b_b79k` indica que el checkpoint fue preentrenado con datos imagen-texto a gran escala. Esto permite usar el modelo como extractor general de representaciones sin necesidad de entrenarlo desde cero. 

En el contexto de semana 4, esto es importante porque el objetivo no es optimizar un modelo fundacional, sino comprender cómo funciona el alineamiento contrastivo y cómo se evalúa la recuperación cruzada.

Desde una perspectiva avanzada, OpenCLIP permite estudiar tres ideas clave:

1. **Representación compartida entre modalidades**  
   Imagen y texto se proyectan al mismo espacio vectorial. Esto hace posible comparar directamente una imagen con muchas captions o una caption con muchas imágenes.

2. **Aprendizaje contrastivo como criterio de alineamiento**  
   Durante el preentrenamiento, el modelo aprende a acercar pares correctos imagen-texto y alejar pares incorrectos. La recuperación que hacemos en este laboratorio es una consecuencia directa de ese entrenamiento.

3. **Uso de modelos preentrenados como baseline experimental**  
   El checkpoint sirve como línea base reproducible. A partir de él se pueden analizar métricas, errores, negativos difíciles, sensibilidad a prompts y diferencias entre datasets sin modificar todavía los pesos del modelo.

En este proyecto, OpenCLIP se usa para construir embeddings normalizados de imágenes y captions. Luego se calcula una matriz de similitud entre ambas modalidades. Esa matriz permite evaluar recuperación imagen->texto y texto->imagen, además de identificar negativos difíciles donde el modelo asigna alta similitud a pares incorrectos pero semánticamente parecidos.

La elección de `ViT-B-32` con `laion2b_s34b_b79k` es apropiada para esta semana porque permite trabajar con un modelo real, abierto y reproducible, manteniendo el costo computacional dentro de lo razonable para una práctica en Docker con una sola GPU.


In [ ]:
model_name = cfg["model"]["model_name"]
pretrained = cfg["model"]["pretrained"]

model, preprocess, tokenizer, device = create_model(model_name, pretrained)
print("device =", device)
print("model_name =", model_name)
print("pretrained =", pretrained)


#### **5. Extraer embeddings del subconjunto bootstrap**


In [ ]:
image_features = encode_image_paths(
    model,
    preprocess,
    metadata["filepath"].tolist(),
    device=device,
    batch_size=cfg["runtime"]["batch_size"],
)

text_features = encode_texts(
    model,
    tokenizer,
    metadata["caption"].tolist(),
    device=device,
    batch_size=max(cfg["runtime"]["batch_size"], 32),
)

image_features.shape, text_features.shape


#### **6. Matriz de similitud y métricas de retrieval**

Como el bootstrap tiene pares alineados `(imagen_i, caption_i)`, la diagonal representa el match correcto.


In [ ]:
sim = image_features @ text_features.T
metrics_i2t = summarize_ranking(sim)
metrics_t2i = summarize_ranking(sim.T)

pd.DataFrame([
    {"direction": "image_to_text", **{k: v for k, v in metrics_i2t.items() if k != "Ranks"}},
    {"direction": "text_to_image", **{k: v for k, v in metrics_t2i.items() if k != "Ranks"}},
])


#### **7. Ejemplos de recuperación cruzada**


In [ ]:
query = "two men cooking in a kitchen"
query_feature = encode_texts(model, tokenizer, [query], device=device)
results = topk_text_to_image(query_feature, image_features, metadata, k=4)
results


In [ ]:
fig = show_retrieval_results(results, figsize=(12, 4))
plt.show()


In [ ]:
img_idx = 3
image_result = topk_image_to_text(image_features[img_idx:img_idx+1], text_features, metadata, k=4)
image_result


#### **8. Negativos difíciles**

Esta parte importa tanto como la métrica agregada. Los negativos difíciles muestran pares incorrectos con score alto y permiten discutir:

- correlaciones espurias,
- ambigüedad semántica,
- similitud superficial,
- límites del embedding compartido.


In [ ]:
hard = mine_hard_negatives(sim, metadata, top_n=8)
hard[["image_id", "image_label", "text_label", "score", "negative_caption"]]


In [ ]:
row = hard.iloc[0]
img = Image.open(row["image_filepath"]).convert("RGB")
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f'Imagen: {row["image_label"]}\nNegative caption score={row["score"]:.3f}')
plt.axis("off")
plt.show()

print("Caption correcto:")
print(row["image_caption"])
print()
print("Negativo dificil:")
print(row["negative_caption"])


#### **9. Zero-shot como extensión**

El proyecto incluye una extensión **zero-shot** mínima para mostrar que un modelo contrastivo imagen-texto sirve también para clasificación sin entrenamiento adicional sobre las clases del laboratorio.

En este contexto, **zero-shot** significa que el modelo no se ajusta con ejemplos etiquetados del bootstrap. En lugar de entrenar un clasificador nuevo, se construyen prompts textuales que representan cada clase posible, por ejemplo una frase del tipo "a photo of a dog" o "a photo of a person riding a bike". Luego, cada imagen se compara contra los embeddings de esos prompts y se asigna la clase cuyo texto queda más cerca en el espacio compartido.

La columna `label` del bootstrap es pequeña y curada solo para demostración. No debe interpretarse como una taxonomía completa de Flickr30k ni como una evaluación robusta de clasificación. Su objetivo es permitir una prueba controlada del mecanismo zero-shot y conectar el laboratorio de recuperación con una tarea de decisión supervisada aparente, pero sin entrenamiento supervisado.

Desde una perspectiva de posgrado, esta extensión permite discutir cuatro ideas importantes:

1. **Clasificación como recuperación texto-imagen**  
   El modelo no aprende una capa clasificadora específica. La predicción surge de comparar la imagen contra descripciones textuales de clases. Por eso, la calidad del resultado depende tanto del encoder visual como de la formulación de los prompts.

2. **Sensibilidad al prompt**  
   Cambiar la redacción de una clase puede modificar la predicción. Por ejemplo, "a photo of a dog", "a dog in an outdoor scene" y "an image of a pet" no necesariamente producen el mismo embedding textual. Esto permite introducir el concepto de prompt engineering en modelos multimodales.

3. **Diferencia entre etiqueta y descripción visual**  
   Una etiqueta corta puede ser demasiado pobre para capturar la escena completa. Si una imagen contiene una persona, una bicicleta, una calle y movimiento, una sola clase puede ocultar ambigüedad semántica. El zero-shot funciona mejor cuando los prompts describen visualmente lo que el modelo puede reconocer.

4. **Evaluación limitada por el diseño del bootstrap**  
   Como el conjunto es pequeño, las métricas zero-shot deben leerse como una demostración funcional. No permiten afirmar que el modelo generaliza bien a una población amplia de imágenes. Para una evaluación más seria se necesitarían más clases, más ejemplos por clase, prompts alternativos y análisis de confusión.

En este proyecto, el zero-shot debe entenderse como una extensión conceptual del alineamiento contrastivo. Primero se aprende un espacio común imagen-texto. Luego se usa ese espacio para recuperar captions, comparar prompts y asignar clases sin reentrenar el modelo.



In [ ]:
!python scripts/02_build_embeddings.py   --metadata-csv data/bootstrap_flickr30k/metadata.csv   --model-name ViT-B-32   --pretrained laion2b_s34b_b79k   --batch-size 16   --output outputs/embeddings/bootstrap_embeddings.npz


In [ ]:
!python scripts/04_eval_zeroshot.py   --embeddings outputs/embeddings/bootstrap_embeddings.npz   --metadata-csv data/bootstrap_flickr30k/metadata.csv   --prompt-config data/bootstrap_flickr30k/prompt_config.json   --output-csv outputs/metrics/zeroshot_predictions.csv


In [ ]:
pd.read_csv(PROJECT_ROOT / "outputs/metrics/zeroshot_predictions.csv")


#### **10. Pipeline reproducible fuera del notebook**

La idea correcta no es dejar todo dentro del notebook.

El proyecto ya trae pipeline reproducible:

- `scripts/run_local_pipeline.sh`
- `scripts/10_train_openclip_csv_local.sh`
- `scripts/11_train_openclip_csv_torchrun.sh`
- `slurm/train_openclip_csv_single_node.sbatch`
- `slurm/train_openclip_csv_multi_node.sbatch`

**Pipeline local completo**
```bash
bash scripts/run_local_pipeline.sh
```

**Fine-tuning CSV local muy corto**
```bash
bash scripts/10_train_openclip_csv_local.sh
```

**Template torchrun**
```bash
bash scripts/11_train_openclip_csv_torchrun.sh
```


#### **11. Descargar un subconjunto mayor y más realista de Flickr30k**

Cuando quieras salir del bootstrap y pasar a algo más serio, ejecuta:


In [ ]:
# Descomenta cuando quieras materializar un subconjunto mayor
# !python scripts/01_prepare_flickr30k_from_hf.py #   --output-root data/processed/flickr1k_hf #   --train-limit 512 #   --val-limit 50 #   --test-limit 50


In [ ]:
!python scripts/01_prepare_flickr30k_from_hf.py \
  --output-root data/processed/flickr1k_hf \
  --train-limit 512 \
  --val-limit 50 \
  --test-limit 50

Después puedes cambiar el CSV de entrada en `scripts/02_build_embeddings.py` y repetir la evaluación.

Eso te deja una progresión clara:

- bootstrap real incluido,
- subconjunto mayor descargado desde HF,
- evaluación reproducible,
- extensión a entrenamiento CSV,
- y plantillas listas para SLURM.


#### **12. Entregables sugeridos**

El estudiante debe entregar un notebook exportado. No se debe crear un archivo Markdown adicional dentro del proyecto.

El informe debe incluir:

1. Comandos ejecutados dentro del contenedor Docker.
2. Evidencia de ejecución del pipeline local con el bootstrap.
3. Evidencia de ejecución del pipeline remoto con `Vishva007/Flickr-Dataset-1k`, si hubo conexión a internet.
4. Tabla comparativa con `R@1`, `R@5`, `R@10` y `MRR`.
5. Análisis comentado de cinco negativos difíciles.
6. Comparación entre `--caption-mode first` y `--caption-mode all`.
7. Una mini ablación experimental.
8. Conclusiones técnicas y limitaciones del modelo.

##### **12.1 Experimentos mínimos**

Ejecutar el pipeline local dentro del contenedor:

```bash
cd /workspace/Semana4/Proyecto
bash scripts/run_local_pipeline.sh
```

Ejecutar el pipeline remoto dentro del contenedor:

```bash
cd /workspace/Semana4/Proyecto
bash scripts/run_hf_flickr1k_pipeline.sh
```

Comparar los resultados generados en:

```text
outputs/metrics/retrieval_metrics.json
outputs/metrics/hard_negatives.csv
outputs/metrics/flickr1k_retrieval_metrics.json
outputs/metrics/flickr1k_hard_negatives.csv
```

##### **12.2 Mini ablación sugerida**

El estudiante debe realizar al menos una de las siguientes variaciones.

**Comparar modos de captions**

Ejecutar una evaluación con una sola caption por imagen:

```bash
python scripts/02_build_embeddings.py \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --model-name ViT-B-32 \
  --pretrained laion2b_s34b_b79k \
  --batch-size 32 \
  --caption-mode first \
  --output outputs/embeddings/flickr1k_embeddings_first.npz

python scripts/03_eval_retrieval.py \
  --embeddings outputs/embeddings/flickr1k_embeddings_first.npz \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --output-json outputs/metrics/flickr1k_retrieval_metrics_first.json \
  --hard-negatives-csv outputs/metrics/flickr1k_hard_negatives_first.csv \
  --top-n-hard-negatives 20
```

Ejecutar una evaluación con todas las captions por imagen:

```bash
python scripts/02_build_embeddings.py \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --model-name ViT-B-32 \
  --pretrained laion2b_s34b_b79k \
  --batch-size 32 \
  --caption-mode all \
  --output outputs/embeddings/flickr1k_embeddings_all.npz

python scripts/03_eval_retrieval.py \
  --embeddings outputs/embeddings/flickr1k_embeddings_all.npz \
  --metadata-csv data/processed/flickr1k_hf/all.csv \
  --output-json outputs/metrics/flickr1k_retrieval_metrics_all.json \
  --hard-negatives-csv outputs/metrics/flickr1k_hard_negatives_all.csv \
  --top-n-hard-negatives 20
```

**Cambiar el tamaño de batch**

Comparar al menos dos valores:

```bash
--batch-size 16
```

```bash
--batch-size 32
```

Reportar si cambia el tiempo de ejecución, el uso de memoria o la estabilidad del proceso.

**Comparar dos checkpoints de OpenCLIP**

Ejemplo base:

```bash
--model-name ViT-B-32
--pretrained laion2b_s34b_b79k
```

Ejemplo alternativo:

```bash
--model-name RN50
--pretrained openai
```

Reportar cuál obtiene mejores métricas y si los errores cualitativos cambian.

##### **12.3 Preguntas para el informe**

Responder brevemente:

1. ¿Qué cambia entre evaluar con una sola caption y evaluar con todas las captions asociadas a la misma imagen?,
2. ¿Qué tipo de errores aparecen en los negativos difíciles?,
3. ¿El modelo confunde objetos, acciones, contexto o relaciones espaciales?,
4. ¿Qué checkpoint obtiene mejores métricas?,
5. ¿Qué limitaciones tiene el bootstrap local frente a Flickr1k?,
6. ¿Qué mejora concreta aplicarías en una siguiente versión del laboratorio?.

##### **12.4 Formato recomendado de entrega**

Toda la documentación del proyecto debe mantenerse en `README.md`.

